In [ ]:
'''
    What are encodings? (এনকোডিং কী?)
    কম্পিউটার সরাসরি "hi" বা "hello" বোঝে না, সে বোঝে বাইনারি বা bytes (যেমন: 01101000)।
    Encoding হলো এমন কিছু নির্দিষ্ট নিয়ম, যা এই raw binary bytes-কে মানুষের পড়ার যোগ্য character বা টেক্সটে (string) রূপান্তর করে।

        UTF-8: এটি হলো বর্তমান স্ট্যান্ডার্ড টেক্সট এনকোডিং। Python 3 ডিফল্টভাবে সব string-কে UTF-8 হিসেবেই রিড করে।

        Mojibake & Unknown Characters: যদি কোনো ডেটা একটি নির্দিষ্ট এনকোডিংয়ে (ধরা যাক, Windows-1252) লেখা থাকে, আর তুমি সেটি অন্য এনকোডিং (যেমন UTF-8) দিয়ে পড়ার চেষ্টা করো, তখন স্ক্রিনে কিছু অর্থহীন বা অদ্ভুত ক্যারেক্টার দেখা যায় (যেমন: æ–‡å—åŒ–ã?? বা ``)। একেই বলে Mojibake বা Character Encoding Mismatch।
'''

In [ ]:
import pandas as pd
import charset_normalizer

# ==========================================
# ধাপ ১: সমস্যার সিমুলেশন (Dataset Creation)
# ==========================================
# ধরি, আমরা একটি ডেটাসেট পেয়েছি যেখানে একটি শহরের নাম জাপানিজ ক্যারেক্টারে লেখা
data = {
    'user_id': [101, 102],
    'name': ['Jihad', 'Taro'],
    'city': ['Dhaka', '東京']  # '東京' মানে টোকিও
}
df_mock = pd.DataFrame(data)

# ফাইলটি আমরা ইচ্ছা করে 'shift_jis' (জাপানিজ এনকোডিং) দিয়ে সেভ করছি।
# রিয়েল লাইফে ক্লায়েন্ট বা সার্ভার থেকে ফাইলটি এভাবেই আসবে।
df_mock.to_csv("japan_data_raw.csv", index=False, encoding="shift_jis")


# ==========================================
# ধাপ ২: সমস্যা (The Error)
# ==========================================
# তুমি যদি সাধারণ নিয়মে এটি রিড করার চেষ্টা করো, Python একে UTF-8 ভাববে এবং ক্র্যাশ করবে।
try:
    df_error = pd.read_csv("japan_data_raw.csv")
except UnicodeDecodeError as e:
    print(f"Error Encountered: {e}\n")
    # Output: 'utf-8' codec can't decode byte 0x93...


# ==========================================
# ধাপ ৩: সমাধান (Detect & Read)
# ==========================================
# ফাইলটির আসল এনকোডিং কী, তা আমরা charset_normalizer দিয়ে বের করবো
with open("japan_data_raw.csv", 'rb') as rawdata:
    # ফাইলের প্রথম ১০০০০ বাইট রিড করে এনকোডিং গেস করা হচ্ছে
    result = charset_normalizer.detect(rawdata.read(10000))

print(f"Detected Encoding: {result['encoding']}") 
# Output: Detected Encoding: shift_jis (বা কাছাকাছি কিছু)

# এবার সঠিক এনকোডিং প্যারামিটারটি দিয়ে ফাইল রিড করবো
df_corrected = pd.read_csv("japan_data_raw.csv", encoding=result['encoding'])

print("\nSuccessfully Read Data:")
print(df_corrected)


# ==========================================
# ধাপ ৪: স্ট্যান্ডার্ডাইজেশন (Save as UTF-8)
# ==========================================
# ডেটা একবার ঠিকভাবে রিড করার পর, আমরা এটিকে গ্লোবাল স্ট্যান্ডার্ড UTF-8 এ সেভ করে রাখবো।
# Pandas ডিফল্টভাবেই utf-8 এ সেভ করে।
df_corrected.to_csv("japan_data_cleaned.csv", index=False)